# Player Recommendation System Experiment

This Colab notebook upgrades the current deterministic player similarity task into a more complete recommendation experiment.

The notebook uses the same clean gold data layer as the rest of the project and keeps the recommendation logic transparent:

- candidate generation with practical scouting filters
- role similarity from standardized feature groups
- optional salary, age, and workload-aware ranking presets
- explainable score breakdowns for each recommendation

The current version is deterministic and exploratory. It does not claim learned ranking quality because the project does not yet have human relevance labels or historical scout feedback.

In [ ]:
# Colab setup: mount Drive, install dependencies, and make project source importable.
from pathlib import Path
import subprocess
import sys

IN_COLAB = Path("/content").exists()
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "nba-scout-assistant"
COLAB_REPO_DIR = Path("/content/nba-scout-assistant")
REPO_URL = "https://github.com/kdnehihi/nba-scout-assistant.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "pyarrow", "scikit-learn"], check=True)

PROJECT_ROOT_CANDIDATES = [
    DRIVE_PROJECT_DIR,
    COLAB_REPO_DIR,
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve(),
]

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "src").exists()), None)
if PROJECT_ROOT is None and IN_COLAB:
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    PROJECT_ROOT = COLAB_REPO_DIR
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project source directory containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = DRIVE_PROJECT_DIR / "data" if IN_COLAB else PROJECT_ROOT / "data"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "reports" / "recommendations" if IN_COLAB else PROJECT_ROOT / "reports" / "recommendations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from src.dataset.loaders import (
    load_performance_training_clean,
    load_role_features_clean,
    load_salary_training_clean,
    resolve_data_paths,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## Load Gold Data

The notebook reads gold data from Drive:

```text
/content/drive/MyDrive/nba-scout-assistant/data/gold
```

In [ ]:
paths = resolve_data_paths(DATA_DIR)

ROLE_PATH = paths.gold_dir / "player_role_features_clean.parquet"
SALARY_PATH = paths.gold_dir / "salary_training_clean.parquet"
PERFORMANCE_PATH = paths.gold_dir / "performance_training_clean.parquet"

for path in [ROLE_PATH, SALARY_PATH, PERFORMANCE_PATH]:
    print(path.name, "exists=", path.exists(), "path=", path)

role_features = load_role_features_clean(paths)
salary = load_salary_training_clean(paths)
performance = load_performance_training_clean(paths)

print("role_features", role_features.shape)
print("salary", salary.shape)
print("performance", performance.shape)

## Feature Groups

The recommender uses feature groups instead of one flat distance only. This makes output easier to explain: the system can say whether a player matched because of scoring, playmaking, rebounding, defense, or workload.

In [ ]:
FEATURE_GROUPS = {
    "workload": [
        "minutes",
        "usage_pct",
    ],
    "scoring": [
        "points_per_100",
        "usage_pct",
        "true_shooting_pct",
        "three_point_attempt_rate",
        "free_throw_rate",
        "scoring_creation",
        "shooting",
        "rim_pressure",
    ],
    "playmaking": [
        "assists_per_100",
        "turnover_rate",
        "playmaking",
    ],
    "rebounding": [
        "rebounds_per_100",
        "defensive_rebound_rate",
        "rebounding",
    ],
    "defense": [
        "steal_rate",
        "block_rate",
        "defensive_rebound_rate",
        "foul_rate",
        "perimeter_defense",
        "interior_defense",
        "two_way_impact",
    ],
}

ALL_RECOMMENDER_FEATURES = sorted({feature for features in FEATURE_GROUPS.values() for feature in features})

RANKING_PRESETS = {
    "role_similarity": {
        "role_similarity_score": 1.00,
        "salary_value_score": 0.00,
        "age_upside_score": 0.00,
        "workload_reliability_score": 0.00,
    },
    "replacement_value": {
        "role_similarity_score": 0.65,
        "salary_value_score": 0.20,
        "age_upside_score": 0.10,
        "workload_reliability_score": 0.05,
    },
    "upside_value": {
        "role_similarity_score": 0.55,
        "salary_value_score": 0.15,
        "age_upside_score": 0.25,
        "workload_reliability_score": 0.05,
    },
    "win_now_fit": {
        "role_similarity_score": 0.65,
        "salary_value_score": 0.10,
        "age_upside_score": 0.05,
        "workload_reliability_score": 0.20,
    },
}

print("features", ALL_RECOMMENDER_FEATURES)
print("ranking presets", RANKING_PRESETS)

## Build Recommendation Base

This section merges player-season role features with salary context and latest short-term form deltas from the performance table.

In [ ]:
def season_start_year(season: object) -> int | None:
    """Input: season label such as 2024-25. Output: season start year."""
    try:
        return int(str(season)[:4])
    except (TypeError, ValueError):
        return None


def position_group(position: object) -> str:
    """Input: raw position. Output: guard, wing, big, or unknown."""
    text = str(position).upper()
    has_g = "G" in text
    has_f = "F" in text
    has_c = "C" in text
    if has_c and not has_g:
        return "big"
    if has_f and not has_c:
        return "wing"
    if has_g and not has_c:
        return "guard"
    if has_f and has_c:
        return "big"
    return "unknown"


def latest_form_by_player_season(performance_df: pd.DataFrame) -> pd.DataFrame:
    """Input: clean performance table. Output: latest form deltas by player-season."""
    latest = (
        performance_df.sort_values(["player_id", "season", "as_of_date", "game_id"])
        .groupby(["player_id", "season"], as_index=False)
        .tail(1)
        .copy()
    )
    cols = [
        "player_id",
        "season",
        "pts_last_5_minus_season_avg",
        "ast_last_5_minus_season_avg",
        "reb_last_5_minus_season_avg",
        "min_last_5_minus_season_avg",
    ]
    return latest[[col for col in cols if col in latest.columns]].reset_index(drop=True)


def build_recommendation_base(
    role_df: pd.DataFrame,
    salary_df: pd.DataFrame,
    performance_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: gold role, salary, performance tables. Output: player-season recommendation base."""
    base = role_df.copy()
    base["season_start_year"] = base["season"].map(season_start_year)
    base["position_group"] = base["position"].map(position_group)

    salary_context = (
        salary_df[["player_id", "season_start_year", "salary_usd", "salary_cap_share", "team_id"]]
        .rename(columns={"team_id": "salary_team_id"})
        .drop_duplicates(["player_id", "season_start_year"])
    )
    form_context = latest_form_by_player_season(performance_df)

    base = base.merge(salary_context, on=["player_id", "season_start_year"], how="left")
    base = base.merge(form_context, on=["player_id", "season"], how="left")

    for column in [*ALL_RECOMMENDER_FEATURES, "salary_cap_share", "age", "minutes"]:
        if column in base.columns:
            base[column] = pd.to_numeric(base[column], errors="coerce")

    return base


recommendation_base = build_recommendation_base(role_features, salary, performance)
print(recommendation_base.shape)
recommendation_base[["player_id", "player_name", "season", "team_id", "position", "position_group", "age", "minutes", "salary_cap_share"]].head()

## Candidate Generation And Scoring

The recommender first filters candidates, then calculates role similarity and optional practical scores. The preset weights are scenario assumptions for exploration, not learned ranking weights.

In [ ]:
def robust_minmax_score(values: pd.Series, higher_is_better: bool = True) -> pd.Series:
    """Input: numeric series. Output: clipped 0-1 score robust to outliers."""
    numeric = pd.to_numeric(values, errors="coerce")
    if numeric.notna().sum() == 0:
        return pd.Series(0.5, index=values.index)
    lower = numeric.quantile(0.05)
    upper = numeric.quantile(0.95)
    if pd.isna(lower) or pd.isna(upper) or lower == upper:
        return pd.Series(0.5, index=values.index)
    clipped = numeric.clip(lower, upper)
    score = (clipped - lower) / (upper - lower)
    if not higher_is_better:
        score = 1 - score
    return score.fillna(0.5)


def select_target_row(base_df: pd.DataFrame, player_name: str, season: str | None = None) -> pd.Series:
    """Input: base table and player query. Output: selected target player-season row."""
    mask = base_df["player_name"].str.contains(player_name, case=False, na=False, regex=False)
    if season is not None:
        mask &= base_df["season"].eq(season)
    matches = base_df[mask].sort_values(["season_start_year", "minutes"], ascending=[False, False])
    if matches.empty:
        raise ValueError(f"No target matched player_name={player_name!r}, season={season!r}")
    return matches.iloc[0]


def generate_candidates(
    base_df: pd.DataFrame,
    target: pd.Series,
    same_season: bool = True,
    same_position_group: bool = True,
    minutes_min: float | None = 500,
    salary_cap_share_max: float | None = None,
    cheaper_only: bool = False,
    younger_only: bool = False,
) -> pd.DataFrame:
    """Input: base table and target row. Output: filtered candidate pool."""
    candidates = base_df[base_df["player_id"].ne(target["player_id"])].copy()
    if same_season:
        candidates = candidates[candidates["season"].eq(target["season"])]
    if same_position_group and pd.notna(target.get("position_group")):
        candidates = candidates[candidates["position_group"].eq(target["position_group"])]
    if minutes_min is not None:
        candidates = candidates[candidates["minutes"].fillna(0) >= minutes_min]
    if salary_cap_share_max is not None:
        candidates = candidates[candidates["salary_cap_share"].fillna(np.inf) <= salary_cap_share_max]
    if cheaper_only and pd.notna(target.get("salary_cap_share")):
        candidates = candidates[candidates["salary_cap_share"].fillna(np.inf) <= float(target["salary_cap_share"])]
    if younger_only and pd.notna(target.get("age")):
        candidates = candidates[candidates["age"].fillna(np.inf) <= float(target["age"])]
    return candidates.reset_index(drop=True)


def standardized_group_distance(scoring_df: pd.DataFrame, target_index: int, features: list[str]) -> pd.Series:
    """Input: scoring dataframe, target row index, features. Output: candidate distances from target."""
    available = [feature for feature in features if feature in scoring_df.columns]
    if not available:
        return pd.Series(np.nan, index=scoring_df.index)
    matrix = scoring_df[available].apply(pd.to_numeric, errors="coerce")
    matrix = matrix.fillna(matrix.median(numeric_only=True)).fillna(0)
    scaled = StandardScaler().fit_transform(matrix)
    target_vector = scaled[target_index]
    distances = np.sqrt(((scaled - target_vector) ** 2).mean(axis=1))
    return pd.Series(distances, index=scoring_df.index)


def add_similarity_scores(target: pd.Series, candidates: pd.DataFrame) -> pd.DataFrame:
    """Input: target and candidate rows. Output: candidates with group and overall similarity scores."""
    scoring_df = pd.concat([target.to_frame().T, candidates], ignore_index=True)
    result = scoring_df.iloc[1:].copy()

    group_distance_cols = []
    for group_name, features in FEATURE_GROUPS.items():
        distances = standardized_group_distance(scoring_df, target_index=0, features=features)
        distance_col = f"{group_name}_distance"
        score_col = f"{group_name}_score"
        result[distance_col] = distances.iloc[1:].to_numpy(dtype="float64")
        result[score_col] = 1 / (1 + result[distance_col])
        group_distance_cols.append(distance_col)

    result["role_similarity_distance"] = result[group_distance_cols].mean(axis=1)
    result["role_similarity_score"] = 1 / (1 + result["role_similarity_distance"])
    return result


def add_practical_scores(target: pd.Series, candidates: pd.DataFrame) -> pd.DataFrame:
    """Input: target and candidates. Output: candidates with salary, age, workload, and gap scores."""
    result = candidates.copy()
    result["salary_value_score"] = robust_minmax_score(result["salary_cap_share"], higher_is_better=False)
    result["age_upside_score"] = robust_minmax_score(result["age"], higher_is_better=False)
    result["workload_reliability_score"] = robust_minmax_score(result["minutes"], higher_is_better=True)
    result["salary_cap_share_gap"] = result["salary_cap_share"] - target.get("salary_cap_share", np.nan)
    result["age_gap"] = result["age"] - target.get("age", np.nan)
    result["minutes_gap"] = result["minutes"] - target.get("minutes", np.nan)
    return result


def add_final_score(candidates: pd.DataFrame, preset: str = "replacement_value") -> pd.DataFrame:
    """Input: scored candidates and preset name. Output: final ranked recommendation score."""
    if preset not in RANKING_PRESETS:
        raise KeyError(f"Unknown ranking preset: {preset}")
    weights = RANKING_PRESETS[preset]
    result = candidates.copy()
    result["ranking_preset"] = preset
    result["recommendation_score"] = 0.0
    for column, weight in weights.items():
        result["recommendation_score"] += weight * result[column].fillna(0.5)
    return result

## Explanations

Each recommendation includes a compact explanation showing the strongest matched role groups and practical gaps versus the target.

In [ ]:
def matched_groups(row: pd.Series, top_n: int = 3) -> str:
    """Input: recommendation row. Output: strongest matching feature groups."""
    score_cols = [f"{group}_score" for group in FEATURE_GROUPS]
    available = [(col.replace("_score", ""), row[col]) for col in score_cols if col in row and pd.notna(row[col])]
    if not available:
        return ""
    ranked = sorted(available, key=lambda item: item[1], reverse=True)[:top_n]
    return ", ".join(f"{name}:{score:.2f}" for name, score in ranked)


def recommendation_reason(row: pd.Series) -> str:
    """Input: recommendation row. Output: readable reason summary."""
    parts = [f"matched groups [{row['matched_groups']}]"]
    if pd.notna(row.get("salary_cap_share_gap")):
        parts.append(f"salary cap share gap {row['salary_cap_share_gap']:+.3f}")
    if pd.notna(row.get("age_gap")):
        parts.append(f"age gap {row['age_gap']:+.1f}")
    if pd.notna(row.get("minutes_gap")):
        parts.append(f"minutes gap {row['minutes_gap']:+.0f}")
    return "; ".join(parts)


def recommend_players(
    base_df: pd.DataFrame,
    player_name: str,
    season: str | None = None,
    top_n: int = 10,
    preset: str = "replacement_value",
    same_season: bool = True,
    same_position_group: bool = True,
    minutes_min: float | None = 500,
    salary_cap_share_max: float | None = None,
    cheaper_only: bool = False,
    younger_only: bool = False,
) -> pd.DataFrame:
    """Input: player query and filters. Output: ranked player recommendations with explanations."""
    target = select_target_row(base_df, player_name=player_name, season=season)
    candidates = generate_candidates(
        base_df=base_df,
        target=target,
        same_season=same_season,
        same_position_group=same_position_group,
        minutes_min=minutes_min,
        salary_cap_share_max=salary_cap_share_max,
        cheaper_only=cheaper_only,
        younger_only=younger_only,
    )
    if candidates.empty:
        raise ValueError("No candidates matched the requested filters.")

    scored = add_similarity_scores(target, candidates)
    scored = add_practical_scores(target, scored)
    scored = add_final_score(scored, preset=preset)
    scored["target_player_name"] = target["player_name"]
    scored["target_player_id"] = target["player_id"]
    scored["target_season"] = target["season"]
    scored["target_salary_cap_share"] = target.get("salary_cap_share", np.nan)
    scored["target_age"] = target.get("age", np.nan)
    scored["matched_groups"] = scored.apply(matched_groups, axis=1)
    scored["recommendation_reason"] = scored.apply(recommendation_reason, axis=1)

    output_cols = [
        "target_player_name",
        "target_player_id",
        "target_season",
        "player_id",
        "player_name",
        "season",
        "team_id",
        "position",
        "position_group",
        "age",
        "minutes",
        "salary_usd",
        "salary_cap_share",
        "recommendation_score",
        "role_similarity_score",
        "salary_value_score",
        "age_upside_score",
        "workload_reliability_score",
        "salary_cap_share_gap",
        "age_gap",
        "minutes_gap",
        "matched_groups",
        "recommendation_reason",
        "ranking_preset",
    ]
    output_cols = [col for col in output_cols if col in scored.columns]
    return scored[output_cols].sort_values("recommendation_score", ascending=False).head(top_n).reset_index(drop=True)

## Run Example Queries

Edit the query variables below for the player and season you want to inspect. If the exact player-season does not exist, the helper selects the most recent matching player-season.

In [ ]:
TARGET_PLAYER = "Desmond Bane"
TARGET_SEASON = "2024-25"
TOP_N = 12

recommendations_role = recommend_players(
    recommendation_base,
    player_name=TARGET_PLAYER,
    season=TARGET_SEASON,
    top_n=TOP_N,
    preset="role_similarity",
    same_position_group=True,
    minutes_min=500,
)

recommendations_value = recommend_players(
    recommendation_base,
    player_name=TARGET_PLAYER,
    season=TARGET_SEASON,
    top_n=TOP_N,
    preset="replacement_value",
    same_position_group=True,
    minutes_min=500,
    cheaper_only=False,
)

print("Role similarity recommendations")
display(recommendations_role)
print("Replacement value recommendations")
display(recommendations_value)

## Compare Ranking Presets

This view shows how recommendations change when the ranking objective changes.

In [ ]:
preset_frames = []
for preset_name in RANKING_PRESETS:
    frame = recommend_players(
        recommendation_base,
        player_name=TARGET_PLAYER,
        season=TARGET_SEASON,
        top_n=8,
        preset=preset_name,
        same_position_group=True,
        minutes_min=500,
    )
    frame.insert(0, "preset", preset_name)
    preset_frames.append(frame)

preset_comparison = pd.concat(preset_frames, ignore_index=True)
preset_comparison[[
    "preset",
    "player_name",
    "team_id",
    "position",
    "age",
    "minutes",
    "salary_cap_share",
    "recommendation_score",
    "role_similarity_score",
    "salary_value_score",
    "age_upside_score",
    "workload_reliability_score",
    "recommendation_reason",
]]

## Candidate Pool Audit

Use this to check whether filters are too strict or too broad.

In [ ]:
target = select_target_row(recommendation_base, TARGET_PLAYER, TARGET_SEASON)
candidate_pool = generate_candidates(
    recommendation_base,
    target=target,
    same_season=True,
    same_position_group=True,
    minutes_min=500,
)

print("target")
display(target[["player_id", "player_name", "season", "team_id", "position", "position_group", "age", "minutes", "salary_cap_share"]].to_frame().T)
print("candidate_pool rows", len(candidate_pool))
print(candidate_pool["position_group"].value_counts(dropna=False))
print(candidate_pool["salary_cap_share"].describe())
print(candidate_pool["minutes"].describe())

## Save Recommendation Output

The output is saved to Drive so it can be reviewed later or compared across query settings.

In [ ]:
output_path = OUTPUT_DIR / f"recommendations_{TARGET_PLAYER.lower().replace(' ', '_')}_{TARGET_SEASON}.csv"
recommendations_value.to_csv(output_path, index=False)
print("Saved", output_path)

## Promotion Notes

Local promotion should happen after reviewing several target players across positions and salary bands. The safest local version should probably expose:

- `role_similarity` as the baseline ranking
- `replacement_value` as a practical scouting ranking
- explanation columns for matched groups and salary/age/workload gaps

The preset weights are not learned yet. To learn ranking weights later, the project needs relevance labels, scout feedback, or a downstream objective such as successful cheaper replacement outcomes.